# Synthetic Multi-Hop Eval Data — Subgraph Sampling (v1)

**Idea (per Neo4j expert):** every `Relation` is linked to the `Chunk` that *asserts* it
(`(:Chunk)-[:ASSERTS]->(:Relation)`). So the graph already gives us the **retrieval ground
truth for free** — for any sampled subgraph we know exactly which **relations**, **entities**,
and **source chunks** *should* be retrieved to answer a question about it.

**Procedure**
1. Load the KG state (`state_closed_v5_merge.pkl`).
2. **Sample a connected subgraph** of ≥2 relations whose evidence spans ≥2 *distinct chunks*
   (a single relation = a single chunk here, so multi-relation is what makes it multi-hop).
3. Ask an LLM to write a **question + expected answer**, grounded only in the evidence, that
   *requires combining* relations from different chunks.
4. **Filter** (deterministic + LLM judge): genuinely multi-hop, faithful, not answerable from
   one chunk, non-trivial.
5. Emit a gold record carrying `gold_chunk_ids / gold_relation_ids / gold_entity_keys` — the
   ground truth used to score the **type_3 PageRank (PPR)** traversal (PIPELINE_v3 Stage 4).

> This is the complement to the RAGAS notebook: RAGAS mines a *chunk-similarity* graph;
> here we mine the *production entity graph PPR actually traverses*, so the gold standard and
> the algorithm share a topology and graph-level metrics (recall@k over entities/relations)
> become possible.

In [24]:
import os
import json
import pickle
import re
import math
import random
import time
from pathlib import Path
from collections import defaultdict, Counter
from dataclasses import dataclass, field
import typing as t

from dotenv import load_dotenv
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_openai import AzureChatOpenAI

load_dotenv()
random.seed(13)

In [25]:
# --- locate repo root (folder that contains output/jupyter-notebook/checkpoints) ---
def find_repo_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / "output" / "jupyter-notebook" / "checkpoints").exists():
            return p
    raise FileNotFoundError("Could not locate repo root with output/jupyter-notebook/checkpoints")

repo_root = find_repo_root(Path.cwd())
print("repo_root =", repo_root)

repo_root = /Users/senghok/Documents/gds_tender_test


In [26]:
# --- LLM client (same APIM pattern as the RAGAS notebook) ---
llm = AzureChatOpenAI(
    azure_deployment=os.getenv("AZURE_OPENAI_VISION_DEPLOYMENT"),
    api_version=os.getenv("AZURE_OPENAI_API_VERSION", "2024-02-15-preview"),
    azure_endpoint=os.getenv("AZURE_OPENAI_APIM"),
    api_key=os.getenv("API_KEY", "DUMMY"),
    default_headers={"Ocp-Apim-Subscription-Key": os.getenv("APIM_SUBSCRIPTION_KEY")},
    temperature=0.4,
)

def parse_json_object(raw: str):
    raw = (raw or "").strip()
    raw = re.sub(r"^```(?:json)?\s*", "", raw, flags=re.I)
    raw = re.sub(r"\s*```$", "", raw)
    m = re.search(r"\{.*\}", raw, flags=re.S)
    return json.loads(m.group(0) if m else raw)

def call_llm_json(system: str, user: str, max_retries: int = 3):
    last = None
    for attempt in range(1, max_retries + 1):
        try:
            resp = llm.invoke([SystemMessage(content=system), HumanMessage(content=user)])
            return parse_json_object(resp.content)
        except Exception as e:
            last = e
            time.sleep(1.5 * attempt)
    raise last

In [17]:
state['all_assertions']

[{'assertion_key': 'f124a87bc8919836fe3df54596ac1837886d911c',
  'chunk_id': 'ROC_INPEX_2_0_Tender_Supplements_WS33071728_-TB-0004_Company_FEED_Value_Enhancement_Optimisation_Update_pdf_size_5858729_page_1_text_chunk_1',
  'relation_id': 'dd5828a6af168af9d6f9ea176c343e4de6df99df',
  'evidence_text': 'Company’s VE process has been developed using the following categorisation waterfall for potential enhancement options: Category 1 Potential opportunities to be developed by Tenderer based on issued Design Dossier (as updated).',
  'confidence': 0.95,
  'relationship_strength': 9,
  'page_number': '1',
  'doc_url': 'file://c:\\Users\\hseng\\Documents\\gds_tender_test\\ROC_INPEX\\2.0 Tender Supplements\\WS33071728 -TB-0004_Company_FEED_Value_Enhancement_Optimisation_Update.pdf',
  'file_name': 'ROC_INPEX/2.0 Tender Supplements/WS33071728 -TB-0004_Company_FEED_Value_Enhancement_Optimisation_Update.pdf'},
 {'assertion_key': '2f391f9d1ddb7a35a32d947206013d7bd51d14ae',
  'chunk_id': 'ROC_INPEX_

In [15]:
chunks_per_rel = defaultdict(set)
for a in state['all_assertions']:
    chunks_per_rel[a['relation_id']].add(a['chunk_id'])
Counter(len(v) for v in chunks_per_rel.values())
# -> {1: 6701, 2: 63, 3: 2}


Counter({1: 6701, 2: 63, 3: 2})

## 1. Load state & build indexes

In [27]:
STATE_FILE = repo_root / "output/jupyter-notebook/checkpoints/state_closed_v5_merge.pkl"
with open(STATE_FILE, "rb") as f:
    state = pickle.load(f)

entity_by_key = {e["entity_key"]: e for e in state["all_entity_nodes"] if e.get("entity_key")}
relation_by_id = {r["relation_id"]: r for r in state["all_relation_nodes"]}
chunk_by_id = {c["chunk_id"]: c for c in state["all_chunk_payloads"]}

# relation_id -> assertions (each carries chunk_id, evidence_text, file, page, confidence)
assertions_by_relation = defaultdict(list)
for a in state["all_assertions"]:
    assertions_by_relation[a["relation_id"]].append(a)

# entity_key -> chunks that MENTION it (used as entity-level gold / PPR seed-recall check)
chunks_by_entity = defaultdict(set)
for m in state["all_mentions"]:
    if m.get("chunk_id") and m.get("entity_key"):
        chunks_by_entity[m["entity_key"]].add(m["chunk_id"])

# undirected adjacency over the reified graph: entity -> [(relation_id, neighbor_entity)]
adj = defaultdict(list)
degree = Counter()
for r in state["all_relation_nodes"]:
    s, o = r["subject_key"], r["object_key"]
    if s not in entity_by_key or o not in entity_by_key:
        continue
    adj[s].append((r["relation_id"], o))
    adj[o].append((r["relation_id"], s))
    degree[s] += 1
    degree[o] += 1

HUB_THRESHOLD = 150          # matches PIPELINE_v3 Practical Defaults
hub_keys = {k for k, d in degree.items() if d >= HUB_THRESHOLD}
print(f"entities={len(entity_by_key)} relations={len(relation_by_id)} chunks={len(chunk_by_id)}")
print(f"hubs(>= {HUB_THRESHOLD}): {len(hub_keys)} -> {[entity_by_key[k]['name'] for k in hub_keys]}")

entities=8965 relations=6766 chunks=1555
hubs(>= 150): 3 -> ['Contractor', 'Tenderer', 'Company']


In [28]:
def relation_chunks(rel_id: str) -> set:
    """Distinct gold chunk ids that ASSERT this relation."""
    return {a["chunk_id"] for a in assertions_by_relation.get(rel_id, []) if a.get("chunk_id")}

def chunk_group(chunk_id: str) -> str:
    """Collapse adjacent chunk-splits of the SAME page to one key, so a single sentence
    split across '..._text_chunk_1' / '_text_chunk_2' is NOT counted as two sources."""
    return re.sub(r"_text_chunk_\d+$", "", chunk_id or "")

def n_distinct_chunks(chunk_ids) -> int:
    """Count genuinely distinct source passages (adjacent splits collapsed)."""
    return len({chunk_group(c) for c in chunk_ids})

def relation_view(rel_id: str) -> dict:
    """Compact human/LLM-readable view of a relation + its evidence."""
    r = relation_by_id[rel_id]
    s, o = entity_by_key[r["subject_key"]], entity_by_key[r["object_key"]]
    asserts = assertions_by_relation.get(rel_id, [])
    return {
        "relation_id": rel_id,
        "subject": s["name"],
        "object": o["name"],
        "label": r.get("relationship_label"),
        "description": r.get("relationship_description"),
        "strength": r.get("relationship_strength"),
        "evidence": [
            {
                "chunk_id": a["chunk_id"],
                "file_name": a.get("file_name"),
                "page": a.get("page_number"),
                "text": (a.get("evidence_text") or "").strip(),
                "confidence": a.get("confidence"),
            }
            for a in asserts
        ],
    }

## 2. Subgraph samplers

Two shapes aligned to the PPR / thematic use-case (PIPELINE_v3 type_3):

- **`sample_path`** — a bridge chain `A -r1- B -r2- C ...` of `n_relations` hops.
  Tests "how do A and C relate" / diffusion across a path.
- **`sample_multiseed`** — grow a small connected neighborhood around a seed (BFS-ish),
  collecting `n_relations` edges. Tests multi-seed thematic diffusion.

**Pre-filters (cheap, before any LLM call):**
- subgraph must span **≥ `min_distinct_chunks`** distinct chunk ids (genuine multi-hop),
- relation evidence confidence ≥ `min_confidence`,
- **hub control**: cap how many endpoints are hubs (`max_hub_nodes`); hub-routed subgraphs are
  *tagged* (`via_hub`) rather than always dropped — keep a few to test the hub penalty.

In [29]:
@dataclass
class Subgraph:
    relation_ids: list
    entity_keys: list
    chunk_ids: list          # union of ASSERTS chunks = retrieval ground truth
    sampler: str
    via_hub: bool

    def n_hops(self): return len(self.relation_ids)
    def n_distinct_chunks(self): return n_distinct_chunks(self.chunk_ids)
    def spans_files(self):
        files = set()
        for rid in self.relation_ids:
            for a in assertions_by_relation.get(rid, []):
                files.add(a.get("file_name"))
        return len(files)

def relation_endpoints(rel_id):
    r = relation_by_id[rel_id]
    return r["subject_key"], r["object_key"]

def join_nodes(rel_ids):
    """Entities shared by >= 2 of the relations -> the 'connectors' of the subgraph.
    A bridge question's join node should be informative (non-hub, not named)."""
    cnt = Counter()
    for rid in rel_ids:
        for k in relation_endpoints(rid):
            cnt[k] += 1
    return {k for k, c in cnt.items() if c >= 2}

def _finalize(rel_ids, sampler):
    rel_ids = list(dict.fromkeys(rel_ids))             # dedup, keep order
    ent_keys, chunk_ids = set(), set()
    for rid in rel_ids:
        s, o = relation_endpoints(rid)
        ent_keys.update([s, o])
        chunk_ids |= relation_chunks(rid)
    via_hub = bool(ent_keys & hub_keys)
    return Subgraph(rel_ids, sorted(ent_keys), sorted(chunk_ids), sampler, via_hub)

In [30]:
def sample_path(n_relations=2, max_tries=200):
    """Random walk producing a chain of n_relations distinct relations."""
    non_hub_seeds = [k for k in adj if k not in hub_keys and degree[k] >= 1]
    for _ in range(max_tries):
        cur = random.choice(non_hub_seeds)
        path, visited_nodes = [], {cur}
        ok = True
        for _ in range(n_relations):
            choices = [(rid, nb) for rid, nb in adj[cur]
                       if rid not in path and nb not in visited_nodes]
            if not choices:
                ok = False
                break
            rid, nb = random.choice(choices)
            path.append(rid)
            visited_nodes.add(nb)
            cur = nb
        if ok and len(path) == n_relations:
            return _finalize(path, "path")
    return None

def sample_multiseed(n_relations=3, max_tries=200):
    """Grow a connected neighborhood: pick a seed, repeatedly add an edge touching the
    current node set. Yields a small induced subgraph (good for thematic/PPR)."""
    seeds = [k for k in adj if k not in hub_keys and 2 <= degree[k] <= 40]
    if not seeds:
        seeds = [k for k in adj if k not in hub_keys]
    for _ in range(max_tries):
        seed = random.choice(seeds)
        frontier = {seed}
        rel_ids = []
        for _ in range(n_relations * 4):
            if len(rel_ids) >= n_relations:
                break
            node = random.choice(list(frontier))
            choices = [(rid, nb) for rid, nb in adj[node] if rid not in rel_ids]
            if not choices:
                continue
            rid, nb = random.choice(choices)
            rel_ids.append(rid)
            frontier.add(nb)
        if len(rel_ids) >= n_relations:
            return _finalize(rel_ids[:n_relations], "multiseed")
    return None

In [31]:
# Quasi-hub connectors: nodes that make a "connection" trivial because almost everything
# attaches to them. Reject subgraphs whose JOIN (connecting) node is one of these or a hub.
STOP_CONNECTOR_NAMES = {
    "project", "the project", "contract", "the contract", "works", "the works",
    "company group", "tender", "contractor", "company", "tenderer",
}
stop_connector_keys = {
    k for k, e in entity_by_key.items()
    if (e.get("name") or "").strip().lower() in STOP_CONNECTOR_NAMES
}
stop_connector_keys |= hub_keys
print("stop-connector entities:", len(stop_connector_keys))

def passes_prefilters(sg: Subgraph, *, min_distinct_chunks=2, max_hub_nodes=1,
                      min_confidence=0.5, block_stop_connectors=True):
    if sg is None:
        return False
    # (fix #2) genuinely multi-hop: count DISTINCT passages, collapsing adjacent splits
    if sg.n_distinct_chunks() < min_distinct_chunks:
        return False
    # hub control: at most N hub entities anywhere in the subgraph
    if len(set(sg.entity_keys) & hub_keys) > max_hub_nodes:
        return False
    # (fix #3) the CONNECTOR must be informative -> reject joins through hub/quasi-hub nodes
    if block_stop_connectors and (join_nodes(sg.relation_ids) & stop_connector_keys):
        return False
    # evidence reliability
    for rid in sg.relation_ids:
        confs = [a.get("confidence", 1.0) for a in assertions_by_relation.get(rid, [])]
        if confs and max(confs) < min_confidence:
            return False
    return True

# quick smoke test
sg = None
while not passes_prefilters(sg):
    sg = sample_path(n_relations=2)
print("sample distinct chunks:", sg.n_distinct_chunks(), "| join nodes:",
      [entity_by_key[k]["name"] for k in join_nodes(sg.relation_ids)])
for rid in sg.relation_ids:
    v = relation_view(rid)
    print(f"  {v['subject']} --[{v['label']}]--> {v['object']}  (chunks={len(v['evidence'])})")

stop-connector entities: 10
sample distinct chunks: 2 | join nodes: ['Shipham']
  Copper alloy (flanged ≤ 150 #) --[approved_vendor]--> Shipham  (chunks=1)
  Shipham --[supplies]--> Gate Valves – Globe valves (ND ≥ 2")  (chunks=1)


## 3. LLM question generation

In [32]:
GEN_SYS = (
    "You generate high-quality MULTI-HOP evaluation questions for a tender/EPC (ITB) "
    "Graph-RAG system. A multi-hop question can only be answered by combining facts from "
    "MORE THAN ONE source chunk. You must ground every claim strictly in the provided evidence."
)

def render_subgraph_for_llm(sg: Subgraph) -> str:
    lines = []
    for i, rid in enumerate(sg.relation_ids, 1):
        v = relation_view(rid)
        lines.append(f"RELATION R{i}: {v['subject']} --[{v['label']}]--> {v['object']}")
        if v["description"]:
            lines.append(f"  meaning: {v['description']}")
        for a in v["evidence"]:
            txt = a["text"][:600]
            lines.append(f"  evidence (chunk={a['chunk_id']} | {a['file_name']} p.{a['page']}): {txt}")
    return "\n".join(lines)

def generate_qa(sg: Subgraph):
    user = f"""You are given a connected subgraph of a tender knowledge graph. Each RELATION is
backed by evidence from a specific source chunk.

{render_subgraph_for_llm(sg)}

Write ONE natural question that a tender analyst might ask, such that:
- answering it REQUIRES combining at least TWO relations that come from DIFFERENT chunks,
- it is NOT answerable from any single chunk alone,
- it does not leak the answer, and is specific (names the real entities, not "R1"/"R2"),
- PREFER a 'bridge' question (the answer requires inferring an intermediate link) over a
  'conjunction' (two unrelated facts about the same entity glued with "and").

Then give the expected answer, grounded ONLY in the evidence above. The answer must resolve
the SPECIFIC connection asked - not merely assert that both endpoints relate to a common node.

Classify the question:
- "bridge"      : needs an intermediate entity/relation to connect the endpoints.
- "conjunction" : two parallel facts about the same named entity.
- "thematic"    : aggregates/summarizes across several relations.

Return STRICT JSON:
{{
  "question": "...",
  "answer": "...",
  "question_type": "bridge|conjunction|thematic",
  "required_relations": ["R1","R2"],   // the relation tags actually needed
  "reasoning": "one line: why it needs multiple chunks",
  "self_multihop": true                // false if you could not avoid single-hop
}}"""
    out = call_llm_json(GEN_SYS, user)
    # map R-tags back to relation_ids / chunks
    tag_to_rid = {f"R{i}": rid for i, rid in enumerate(sg.relation_ids, 1)}
    req = [tag_to_rid[t] for t in out.get("required_relations", []) if t in tag_to_rid]
    out["required_relation_ids"] = req or sg.relation_ids
    out["required_chunk_ids"] = sorted({c for rid in out["required_relation_ids"]
                                        for c in relation_chunks(rid)})
    return out

## 4. Deterministic + LLM-judge filtering

In [33]:
JUDGE_SYS = (
    "You are a strict validator of synthetic MULTI-HOP QA used to evaluate a tender Graph-RAG "
    "system. Use ONLY the provided evidence."
)

def judge_qa(sg: Subgraph, qa: dict):
    user = f"""EVIDENCE (grouped by relation/chunk):
{render_subgraph_for_llm(sg)}

QUESTION: {qa['question']}
ANSWER:   {qa['answer']}

Score each 0-5 and decide acceptance. A good item is faithful, genuinely multi-hop
(needs >=2 chunks), specific, non-trivial, and the answer must RESOLVE the connection asked.

Return STRICT JSON:
{{
  "faithfulness": 0-5,         // answer fully supported by evidence, nothing invented
  "multihop_necessity": 0-5,   // truly needs >=2 chunks (0 if one chunk suffices)
  "answer_completeness": 0-5,  // resolves the SPECIFIC connection asked, not just "both relate to X"
  "clarity": 0-5,
  "accept": true/false,
  "reason": "one line"
}}"""
    return call_llm_json(JUDGE_SYS, user)

def deterministic_ok(sg: Subgraph, qa: dict, min_required_chunks=2):
    # the relations the LLM says are required must span >=2 DISTINCT chunks (splits collapsed)
    return n_distinct_chunks(qa.get("required_chunk_ids", [])) >= min_required_chunks

## 5. Orchestration — sample → generate → validate → collect

In [34]:
def build_dataset(n_target=50, samplers=None, judge_threshold=4, max_attempts=None):
    samplers = samplers or [
        ("path", lambda: sample_path(n_relations=random.choice([2, 3]))),
        ("multiseed", lambda: sample_multiseed(n_relations=random.choice([3, 4]))),
    ]
    max_attempts = max_attempts or n_target * 12
    rows, attempts, seen_questions = [], 0, set()

    while len(rows) < n_target and attempts < max_attempts:
        attempts += 1
        name, fn = random.choice(samplers)
        sg = fn()
        if not passes_prefilters(sg):
            continue
        try:
            qa = generate_qa(sg)
        except Exception as e:
            print("gen error:", e); continue
        if not qa.get("self_multihop", True) or not deterministic_ok(sg, qa):
            continue
        q_norm = re.sub(r"\W+", " ", qa["question"].lower()).strip()
        if q_norm in seen_questions:
            continue
        try:
            verdict = judge_qa(sg, qa)
        except Exception as e:
            print("judge error:", e); continue
        # gate on faithfulness, multi-hop necessity AND answer completeness (fix #5)
        gate = [verdict.get("faithfulness", 0),
                verdict.get("multihop_necessity", 0),
                verdict.get("answer_completeness", 0)]
        if not verdict.get("accept") or min(gate) < judge_threshold:
            continue

        # (fix #4) derive ALL gold + metadata from the REQUIRED subgraph, not the full sample,
        # so gold_entity_keys / n_hops / via_hub describe exactly what the question needs.
        req_sg = _finalize(qa["required_relation_ids"], sg.sampler)

        seen_questions.add(q_norm)
        rows.append({
            "question": qa["question"],
            "reference_answer": qa["answer"],
            "question_type": qa.get("question_type"),
            # ---- retrieval ground truth (REQUIRED relations only) ----
            "gold_chunk_ids": req_sg.chunk_ids,
            "gold_relation_ids": req_sg.relation_ids,
            "gold_entity_keys": req_sg.entity_keys,
            "gold_entity_names": [entity_by_key[k]["name"] for k in req_sg.entity_keys],
            # ---- provenance / metadata (also from REQUIRED subgraph) ----
            "sampler": sg.sampler,
            "n_hops": req_sg.n_hops(),
            "n_distinct_chunks": req_sg.n_distinct_chunks(),
            "spans_files": req_sg.spans_files(),
            "via_hub": req_sg.via_hub,
            # ---- full sampled subgraph kept for transparency/debugging ----
            "sampled_relation_ids": sg.relation_ids,
            "sampled_chunk_ids": sg.chunk_ids,
            "judge": verdict,
            "gen_reasoning": qa.get("reasoning"),
        })
        if len(rows) % 5 == 0:
            print(f"  collected {len(rows)}/{n_target} (attempts={attempts})")
    print(f"DONE: {len(rows)} rows in {attempts} attempts")
    return rows

# dataset = build_dataset(n_target=50)   # uncomment to run

In [35]:
# Run a small batch first to sanity-check quality, then scale up.
dataset = build_dataset(n_target=10)

  collected 5/10 (attempts=16)
  collected 10/10 (attempts=33)
DONE: 10 rows in 33 attempts


In [36]:
dataset

[{'question': 'What milestone is scheduled to occur at the same time as the completion of the AGRU Lean Solvent Booster Pump L-552-P-001-A, and what approval does it require?',
  'reference_answer': 'The milestone FG3-8 (All sub-systems associated within CCS Pipeline achieve Company’s Approval) is scheduled to occur at CD+ xx Months, the same time as the completion of the AGRU Lean Solvent Booster Pump L-552-P-001-A. FG3-8 requires Ready for Start-up Certificates for Company’s Approval.',
  'question_type': 'bridge',
  'gold_chunk_ids': ['ROC_INPEX_ITT_Ichthys_Onshore_AGRU_Upgrade_and_CCS_EPC_Contract_WS33071728_ITT_Part_B_-_Input_Exhibit_C_-_Work_Time_Schedule_Rev0_20241216_xlsx_size_33325_sheet_Exhibit_C_chunk_17',
   'ROC_INPEX_ITT_Ichthys_Onshore_AGRU_Upgrade_and_CCS_EPC_Contract_WS33071728_ITT_Part_B_-_Input_Exhibit_C_-_Work_Time_Schedule_Rev0_20241216_xlsx_size_33325_sheet_Exhibit_C_chunk_22'],
  'gold_relation_ids': ['48668b3fb475b70ebcd29c71576bf3a21ec79451',
   'c0a6d3c19f031a

In [ ]:

for r in dataset[:3]:
    print("Q:", r["question"])
    print("A:", r["reference_answer"][:200])
    print(f"   type={r['question_type']} | gold_chunks={len(r['gold_chunk_ids'])}"
          f" (distinct={r['n_distinct_chunks']}) | hops={r['n_hops']}"
          f" | via_hub={r['via_hub']} | sampler={r['sampler']}")
    print()

In [ ]:
OUTPUT_PATH = repo_root / "chatbot" / "data" / "synthetic_qa_subgraph_multihop_v1.jsonl"
with OUTPUT_PATH.open("w", encoding="utf-8") as f:
    for r in dataset:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")
print("wrote", len(dataset), "rows ->", OUTPUT_PATH)

## 6. How to score the PageRank (type_3) traversal with this set

Because the gold standard lives in the **same graph PPR traverses**, you get graph-level
metrics the RAGAS chunk set cannot give:

```python
# Pseudo-code — wire to your type_3 answer_thematic(question)
def evaluate(dataset, run_ppr):
    chunk_recall, ent_recall, rr = [], [], []
    for row in dataset:
        result = run_ppr(row["question"])     # -> {"chunk_ids":[...], "entity_keys":[...ranked]}
        gold_c = set(row["gold_chunk_ids"])
        gold_e = set(row["gold_entity_keys"])
        ret_c  = set(result["chunk_ids"])
        chunk_recall.append(len(gold_c & ret_c) / len(gold_c))
        ent_recall.append(len(gold_e & set(result["entity_keys"])) / len(gold_e))
        # MRR of first gold entity in the PPR ranking
        ranks = [i for i, k in enumerate(result["entity_keys"], 1) if k in gold_e]
        rr.append(1.0 / ranks[0] if ranks else 0.0)
    return {"chunk_recall": mean(chunk_recall),
            "entity_recall": mean(ent_recall),
            "entity_MRR": mean(rr)}
```

Useful slices: `via_hub` (does the hub penalty help or hurt?), `n_hops` (does recall decay with
hop count?), `sampler` (path vs multiseed), `spans_files` (cross-document diffusion).